In [2]:
import torch
import pandas as pd
import re
import random
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.next_token_dataset import build_vocab, NextTokenDataset, collate_fn
from src.lstm_model import LSTMModel
from src.lstm_train import train_epoch, evaluate
from src.eval_lstm import evaluate_rouge
from src.eval_transformer_pipeline import evaluate_transformer_rouge

random.seed(42)
torch.manual_seed(42)

In [3]:
# очистка и сохранение данных
from src.data_utils import clean_string

BASE = "/home/rays/Загрузки/Projects/text-autocomplete"

with open(f"{BASE}/data/tweets.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

pd.DataFrame({"text": raw_lines}).to_csv(f"{BASE}/data/raw_dataset.csv", index=False)
print(f"Сырых строк: {len(raw_lines)}")

seq_len = 7
cleaned_texts = [clean_string(line) for line in raw_lines]
cleaned_texts = [t for t in cleaned_texts if len(t.split()) >= seq_len]
pd.DataFrame({"text": cleaned_texts}).to_csv(f"{BASE}/data/dataset_processed.csv", index=False)
print(f"После очистки: {len(cleaned_texts)}")

train_val, test_texts = train_test_split(cleaned_texts, test_size=0.1, random_state=42)
train_texts, val_texts = train_test_split(train_val, test_size=0.1/0.9, random_state=42)

pd.DataFrame({"text": train_texts}).to_csv(f"{BASE}/data/train.csv", index=False)
pd.DataFrame({"text": val_texts}).to_csv(f"{BASE}/data/val.csv", index=False)
pd.DataFrame({"text": test_texts}).to_csv(f"{BASE}/data/test.csv", index=False)

total = len(train_texts) + len(val_texts) + len(test_texts)
print(f"Train: {len(train_texts)} ({len(train_texts)/total*100:.1f}%)")
print(f"Val:   {len(val_texts)}  ({len(val_texts)/total*100:.1f}%)")
print(f"Test:  {len(test_texts)}  ({len(test_texts)/total*100:.1f}%)")


Сырых строк: 1600498
После очистки: 1272840
Train: 1018271 (80.0%)
Val:   127285  (10.0%)
Test:  127284  (10.0%)


In [4]:
# датасеты и загрузчики
vocab = build_vocab(train_texts, min_freq=5)
print(f"Словарь: {len(vocab)} слов")

train_dataset = NextTokenDataset(train_texts, vocab)
val_dataset   = NextTokenDataset(val_texts,   vocab)
test_dataset  = NextTokenDataset(test_texts,  vocab)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, collate_fn=collate_fn)

x, y = next(iter(train_loader))
print(f"Batch X: {x.shape}, Batch Y: {y.shape}")

Словарь: 65065 слов
Batch X: torch.Size([128, 27]), Batch Y: torch.Size([128, 27])


In [8]:
torch.cuda.empty_cache()

In [9]:
import importlib
import src.lstm_model
importlib.reload(src.lstm_model)
from src.lstm_model import LSTMModel

torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = LSTMModel(len(vocab), embed_dim=64, hidden_dim=128).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

model_path = f"{BASE}/models/lstm.pt"

try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Найдена сохранённая модель, загружаем веса")
except FileNotFoundError:
    print("Сохранённой модели нет, обучаем с нуля")
    for epoch in range(10):
        train_loss = train_epoch(model, train_loader, optimizer, device, len(vocab))
        val_loss   = evaluate(model, val_loader, device, len(vocab))
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}")
    torch.save(model.state_dict(), model_path)
    print("Модель сохранена")


Device: cuda
Найдена сохранённая модель, загружаем веса


/tmp/ipykernel_664/1884025371.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


In [18]:
# ROUGE LSTM
print("=== LSTM ===")
_ = evaluate_rouge(model, val_dataset, vocab, device)


=== LSTM ===
ROUGE-1: 0.0641
ROUGE-2: 0.0070


In [17]:
from transformers import logging
logging.set_verbosity_error()

# ROUGE трансформер
print("=== distilgpt2 ===")
_ = evaluate_transformer_rouge(val_dataset, vocab)


=== distilgpt2 ===


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

ROUGE-1: 0.0381
ROUGE-2: 0.0029


In [20]:
# финальный тест лучшей модели
print("=== Финальный тест ===")
_ = evaluate_rouge(model, test_dataset, vocab, device)


=== Финальный тест ===
ROUGE-1: 0.0642
ROUGE-2: 0.0084
